[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_5_aruco/14_aruco_pose_estimation.ipynb)

# Notebook 14 — ArUco Pose Estimation

**Part 5: ArUco Markers** | Estimated time: 50 min

---

We have detected corners + IDs. Now we compute **full 6D pose** — the position and orientation of the marker relative to the camera.

```
Pipeline position:

  Generate  →  Detect corners  →  [Estimate rvec, tvec]  →  Draw axes / robotics
                                         ← YOU ARE HERE
```

This is the step where everything comes together:  
**Camera calibration** (K, dist) + **detected corners** → `estimatePoseSingleMarkers` → **6D pose**

## Recommended Watch

> Watch **VN14 first** — it's a focused, dedicated walkthrough of ArUco pose estimation. Then optionally re-watch the pose section of the NB 11 video for the end-to-end context.

| Title | Link | Duration | Note |
|---|---|---|---|
| **ArUco Marker Pose Estimation in OpenCV** | [▶ Watch](https://youtu.be/GEWoGDdjlSc?si=HcQ-ULlZQzvc0GC8) | shorter | Dedicated pose estimation walkthrough — watch this first |
| **ArUco Marker Pose Estimation and Detection in Real-Time using OpenCV Python** | [▶ Watch](https://www.youtube.com/watch?v=bS00Vs09Upw) | ~25 min | Re-watch the pose section for end-to-end context |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python matplotlib numpy -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Explain why camera calibration is required for pose estimation
- Call `estimatePoseSingleMarkers` and interpret the outputs
- Draw coordinate frame axes on a detected marker
- Extract physical distance from `tvec`
- Convert `rvec` to a rotation matrix and understand what it means
- Handle multiple markers simultaneously

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'OpenCV version: {cv2.__version__}')

def show_bgr(img, title='', figsize=(9, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 1. Why Camera Calibration Is Required

Pose estimation answers: *"Where is the marker in 3D space relative to the camera?"*

To answer this, we need to go from **2D image pixels** back to **3D world coordinates** — the inverse of the projection equation:

$$\mathbf{x} = K[R|t]\mathbf{X}$$

Without knowing $K$ (focal length, principal point), we cannot invert this equation — we have no idea how pixels map to angles.

**What calibration gives us:**

| Parameter | Meaning | Why needed |
|---|---|---|
| $f_x, f_y$ | Focal lengths in pixels | Converts pixel offsets to angles |
| $c_x, c_y$ | Principal point | Defines the optical center |
| $k_1, k_2, p_1, p_2, k_3$ | Distortion coefficients | Undistorts corner positions before estimation |

In [ ]:
# Load calibration saved in notebook 07 (or use realistic defaults)
CALIB_PATH = '../assets/calibration/camera_calibration.npz'

if os.path.exists(CALIB_PATH):
    data = np.load(CALIB_PATH)
    K    = data['camera_matrix']
    dist = data['dist_coeffs']
    print(f'Loaded calibration from {CALIB_PATH}')
else:
    # Realistic defaults for a 640×480 webcam
    K = np.array([
        [600, 0,   320],
        [0,   600, 240],
        [0,   0,   1  ]
    ], dtype=np.float64)
    dist = np.zeros((1, 5), dtype=np.float64)
    print('Using default calibration (no .npz found)')
    print('For best results, run notebook 07 first!')

print(f'\nCamera Matrix K:')
print(K)
print(f'\nDistortion Coefficients:')
print(dist)

## 2. The `estimatePoseSingleMarkers` API

`estimatePoseSingleMarkers` is the single function that bridges detection and pose. It takes the corners you found in NB 13 — combined with the known physical size of the marker and the camera's intrinsic matrix — and returns the **3D position and orientation** of each detected marker in camera space.

The key input is `marker_length`: the real-world side length of your marker in **meters**. If your printed marker is 15 cm, pass `0.15`. Get this wrong and every distance measurement will be off by the same factor (tvec is in the same units you provide).

```python
rvecs, tvecs, objPoints = cv2.aruco.estimatePoseSingleMarkers(
    corners,        # list of (1,4,2) corner arrays from detectMarkers
    marker_length,  # real-world marker side length in METERS
    cameraMatrix,   # K — 3×3 intrinsic matrix
    distCoeffs      # distortion coefficients
)
```

**Important:** `marker_length` is in meters. If your marker is 15 cm, pass `0.15`.

| Output | Shape | Meaning |
|---|---|---|
| `rvecs` | `(N, 1, 3)` | Rodrigues rotation vector per marker |
| `tvecs` | `(N, 1, 3)` | Translation vector per marker (meters) |
| `objPoints` | `(N, 4, 3)` | 3D corners in marker coordinate frame |

The marker coordinate frame is defined as:
```
Marker face-on:

  (-L/2,  L/2, 0)  ────  ( L/2,  L/2, 0)    where L = marker_length
        │                       │
  (-L/2, -L/2, 0)  ────  ( L/2, -L/2, 0)
                                              Z axis points OUT of the marker toward you
```

## 3. Setting Up the Test Scene

We create a synthetic scene where we know the true pose of each marker, then verify that `estimatePoseSingleMarkers` recovers it correctly.

In [ ]:
# API compatibility helpers (same as notebook 13)
def get_aruco_dict(dict_id):
    try:
        return cv2.aruco.getPredefinedDictionary(dict_id)
    except AttributeError:
        return cv2.aruco.Dictionary_get(dict_id)

def generate_marker(aruco_dict, marker_id, size_px):
    try:
        return cv2.aruco.generateImageMarker(aruco_dict, marker_id, size_px)
    except AttributeError:
        img = np.zeros((size_px, size_px, 1), dtype='uint8')
        cv2.aruco.drawMarker(aruco_dict, marker_id, size_px, img, 1)
        return img.squeeze()

def detect_markers_api(image, aruco_dict_obj):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    try:
        params = cv2.aruco.DetectorParameters()
        params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
        detector = cv2.aruco.ArucoDetector(aruco_dict_obj, params)
        return detector.detectMarkers(gray)
    except AttributeError:
        params = cv2.aruco.DetectorParameters_create()
        return cv2.aruco.detectMarkers(gray, aruco_dict_obj, parameters=params)

# Create a clean marker on a white background
DICT_ID       = cv2.aruco.DICT_4X4_50
MARKER_ID     = 0
MARKER_LENGTH = 0.10  # 10 cm marker
MARKER_PX     = 400

aruco_dict_obj = get_aruco_dict(DICT_ID)

# Generate a marker on a white background (simulates a flat scene)
scene_size = 640
scene = np.ones((scene_size, scene_size, 3), dtype=np.uint8) * 220

marker_gray = generate_marker(aruco_dict_obj, MARKER_ID, MARKER_PX)
marker_bgr = cv2.cvtColor(marker_gray, cv2.COLOR_GRAY2BGR)

# Place marker centered in scene
x0 = (scene_size - MARKER_PX) // 2
y0 = (scene_size - MARKER_PX) // 2
scene[y0:y0+MARKER_PX, x0:x0+MARKER_PX] = marker_bgr

show_bgr(scene, f'Test scene — marker ID={MARKER_ID}, L={MARKER_LENGTH*100:.0f} cm')

In [ ]:
# Detect markers
corners, ids, rejected = detect_markers_api(scene, aruco_dict_obj)

print(f'Detected: {len(ids.flatten()) if ids is not None else 0} markers')

# Estimate pose
if ids is not None:
    rvecs, tvecs, obj_pts = cv2.aruco.estimatePoseSingleMarkers(
        corners,
        MARKER_LENGTH,
        K,
        dist
    )
    
    print('\n=== Pose Estimation Results ===')
    for i, (mid, rvec, tvec) in enumerate(zip(ids.flatten(), rvecs, tvecs)):
        print(f'\nMarker ID={mid}:')
        print(f'  rvec (Rodrigues): {rvec.flatten().round(4)}')
        print(f'  tvec (meters):    {tvec.flatten().round(4)}')
        
        # Distance from camera
        distance = np.linalg.norm(tvec)
        print(f'  Distance from camera: {distance:.3f} m = {distance*100:.1f} cm')
        
        # X/Y offset (lateral position)
        tx, ty, tz = tvec.flatten()
        print(f'  Camera→Marker offset: X={tx:.3f}m, Y={ty:.3f}m, Z={tz:.3f}m (depth)')
else:
    print('No markers detected — cannot estimate pose')

## 4. Understanding tvec — Translation Vector

$$\text{tvec} = \begin{bmatrix} t_X \\ t_Y \\ t_Z \end{bmatrix}$$

Where:
- $t_X$ = horizontal offset from camera optical axis (positive = right)
- $t_Y$ = vertical offset (positive = down in image coordinates)
- $t_Z$ = **depth** — distance along camera's optical axis (**the most useful value for robotics!**)

```
Camera view:

    Camera  ────── Z axis ──────►  [marker at distance t_Z]
       │
       │ t_X = left/right offset
       │ t_Y = up/down offset
```

> **Units:** tvec is in the same units as `marker_length`. If you pass `0.10` (10 cm), tvec is in meters.

In [ ]:
if ids is not None:
    tvec = tvecs[0].flatten()
    tx, ty, tz = tvec
    distance = np.linalg.norm(tvec)
    
    print('=== tvec Interpretation ===')
    print(f'tvec = [{tx:.4f}, {ty:.4f}, {tz:.4f}] meters')
    print()
    print(f'X (left/right):  {tx*100:+.1f} cm  (+ = marker to the right)')
    print(f'Y (up/down):     {ty*100:+.1f} cm  (+ = marker is below center)')
    print(f'Z (depth):       {tz*100:.1f} cm  (distance along camera axis)')
    print(f'Euclidean dist:  {distance*100:.1f} cm')
    print()
    print('→ For robotics alignment: drive until tx ≈ 0 (centered) and tz = target_distance')

## 5. Understanding rvec — Rotation Vector (Rodrigues)

The rotation vector $\mathbf{r}$ encodes a 3D rotation in compact form:

$$\mathbf{r} = \theta \hat{\mathbf{n}}$$

Where:
- $\hat{\mathbf{n}}$ = **unit vector** pointing along the rotation axis  
- $\theta = |\mathbf{r}|$ = **rotation angle** in radians

**Example:** `rvec = [0, 0, 1.57]` means "rotate 90° around the Z axis"

Convert to rotation matrix with `cv2.Rodrigues(rvec)`:

In [ ]:
if ids is not None:
    rvec = rvecs[0]
    
    # Convert to rotation matrix
    R, _ = cv2.Rodrigues(rvec)
    
    print('=== rvec Interpretation ===')
    print(f'rvec = {rvec.flatten().round(4)} rad')
    
    angle = np.linalg.norm(rvec)
    axis  = rvec.flatten() / (angle + 1e-10)
    print(f'Rotation angle: {np.degrees(angle):.1f} degrees')
    print(f'Rotation axis:  {axis.round(4)}')
    
    print(f'\nRotation matrix R (3×3):')
    print(R.round(4))
    
    # Verify it's a valid rotation matrix
    RRT = R @ R.T
    det = np.linalg.det(R)
    print(f'\nValidation:')
    print(f'  R @ R.T ≈ I? Max error: {np.abs(RRT - np.eye(3)).max():.6f}')
    print(f'  det(R) = {det:.6f} (should be +1.0)')
    
    # Extract approximate Euler angles (roll/pitch/yaw)
    # Note: Euler angles have gimbal lock issues, use with caution
    pitch = np.degrees(np.arctan2(-R[2,0], np.sqrt(R[0,0]**2 + R[1,0]**2)))
    yaw   = np.degrees(np.arctan2(R[1,0], R[0,0]))
    roll  = np.degrees(np.arctan2(R[2,1], R[2,2]))
    print(f'\nApproximate Euler angles (ZYX convention):')
    print(f'  Roll={roll:.1f}°  Pitch={pitch:.1f}°  Yaw={yaw:.1f}°')

## 6. Drawing Coordinate Frame Axes

`cv2.drawFrameAxes` projects the 3D coordinate frame origin and axis tips onto the image and draws colored lines:

- **Red** = X axis (points right along marker face)
- **Green** = Y axis (points down along marker face) 
- **Blue** = Z axis (points OUT toward the camera)

```python
cv2.drawFrameAxes(
    image,          # drawn on this image
    cameraMatrix,   # K
    distCoeffs,     # dist
    rvec,           # rotation (for this marker)
    tvec,           # translation (for this marker)
    length          # axis length in same units as marker_length
)
```

In [ ]:
def draw_aruco_pose(image, corners, ids, rvecs, tvecs, K, dist,
                    marker_length, axis_length=None):
    """
    Draw detected markers + 3D coordinate frame axes.
    
    Args:
        axis_length: length of drawn axes (default = half marker_length)
    """
    if axis_length is None:
        axis_length = marker_length * 0.5
    
    output = image.copy()
    
    if ids is None:
        cv2.putText(output, 'No markers detected', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 2)
        return output
    
    # Draw marker outlines
    cv2.aruco.drawDetectedMarkers(output, corners, ids)
    
    for i, (mid, rvec, tvec) in enumerate(zip(ids.flatten(), rvecs, tvecs)):
        # Draw coordinate frame axes
        cv2.drawFrameAxes(output, K, dist, rvec, tvec, axis_length)
        
        # Add pose text
        tx, ty, tz = tvec.flatten()
        dist_m = np.linalg.norm(tvec)
        
        pts = corners[i].reshape(4, 2).astype(int)
        label_x = pts[0, 0]  # top-left corner x
        label_y = pts[0, 1] - 15  # above marker
        
        cv2.putText(output,
                    f'ID={mid} | d={dist_m*100:.1f}cm',
                    (label_x, label_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (0, 255, 0), 2)
        
        cv2.putText(output,
                    f'X={tx*100:+.1f} Y={ty*100:+.1f} Z={tz*100:.1f}cm',
                    (label_x, label_y + 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                    (255, 255, 0), 1)
    
    return output

# Draw results
annotated = draw_aruco_pose(scene, corners, ids, rvecs, tvecs, K, dist,
                             MARKER_LENGTH)
show_bgr(annotated, 'ArUco pose — axes: Red=X, Green=Y, Blue=Z (pointing toward camera)')

## 7. Multiple Markers Simultaneously

`estimatePoseSingleMarkers` processes all detected markers in one call and returns one rvec/tvec per marker.  
This works because each marker independently provides 4 point correspondences.

In [ ]:
# Create a scene with multiple markers at different distances (simulated)
def create_multi_marker_scene():
    scene = np.ones((600, 900, 3), dtype=np.uint8) * 210
    aruco_d = get_aruco_dict(cv2.aruco.DICT_4X4_50)
    
    # Three markers at different apparent sizes (simulating different distances)
    configs = [
        {'id': 0, 'px': 200, 'pos': (50,  50)},   # large = close
        {'id': 1, 'px': 140, 'pos': (330, 80)},   # medium
        {'id': 2, 'px': 100, 'pos': (590, 100)},  # small = far
    ]
    
    for cfg in configs:
        m = generate_marker(aruco_d, cfg['id'], cfg['px'])
        m_bgr = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        x0, y0 = cfg['pos']
        x1, y1 = x0 + cfg['px'], y0 + cfg['px']
        if y1 <= scene.shape[0] and x1 <= scene.shape[1]:
            scene[y0:y1, x0:x1] = m_bgr
    
    return scene

multi_scene = create_multi_marker_scene()

# Detect and estimate
c_multi, i_multi, r_multi = detect_markers_api(multi_scene, aruco_dict_obj)

if i_multi is not None:
    rv_multi, tv_multi, _ = cv2.aruco.estimatePoseSingleMarkers(
        c_multi, MARKER_LENGTH, K, dist)
    
    print(f'Detected {len(i_multi.flatten())} markers')
    for mid, tvec in zip(i_multi.flatten(), tv_multi):
        tz = tvec.flatten()[2]
        print(f'  Marker ID={mid}: depth Z = {tz*100:.1f} cm')
    
    annotated_multi = draw_aruco_pose(
        multi_scene, c_multi, i_multi, rv_multi, tv_multi, K, dist, MARKER_LENGTH)
    show_bgr(annotated_multi, 'Multiple markers — each gets independent pose estimate',
             figsize=(12, 7))

## 8. Real-Time Pose Estimation Script

Complete script for live webcam pose estimation. Save as `aruco_pose_live.py` and run locally.

In [ ]:
# The real-time pose estimation script lives in scripts/aruco/aruco_pose_estimation_live.py
# Run it from the repo root in a terminal:
#
#   python scripts/aruco/aruco_pose_estimation_live.py
#
# Optional arguments:
#   --calib          PATH    calibration .npz from NB07 (default: auto-detect)
#   --marker-length  FLOAT   marker side in METERS — measure your printed marker! (default: 0.10)
#   --dict           NAME    ArUco dictionary (default: DICT_4X4_50)
#   --camera         INT     webcam index (default: 0)
#
# Controls while running:
#   Q / Esc   → quit
#
# Draws Red=X, Green=Y, Blue=Z axes on each detected marker.
# Overlays distance (cm) and X/Y offset per marker.

print("See scripts/aruco/aruco_pose_estimation_live.py — run from repo root in a terminal.")

## Recap

| Concept | Key Point |
|---|---|
| `estimatePoseSingleMarkers` | Takes corners + K + dist → rvecs, tvecs |
| `marker_length` | Real-world side in **meters** — must be accurate |
| `tvec` | [X, Y, Z] in meters — Z is depth from camera |
| `rvec` | Rodrigues vector — angle × axis |
| `cv2.Rodrigues(rvec)` | Converts to 3×3 rotation matrix |
| `cv2.drawFrameAxes` | Red=X, Green=Y, Blue=Z pointing toward camera |
| Units | All in meters if `marker_length` is in meters |

**Next notebook:** Full robotics application — use the pose to compute docking alignment offsets.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Change marker size and observe tvec change
# ============================================================
# Using the same scene, estimate pose with MARKER_LENGTH = 0.05 (5 cm)
# and again with MARKER_LENGTH = 0.20 (20 cm).
# How does tvec.Z change? Why?
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# for L in [0.05, 0.10, 0.20]:
#     rv, tv, _ = cv2.aruco.estimatePoseSingleMarkers(corners, L, K, dist)
#     if tv is not None:
#         tz = tv[0].flatten()[2]
#         print(f'marker_length={L:.2f}m → Z={tz*100:.1f}cm')
# # Z scales proportionally with marker_length!
# # If you double marker_length, Z doubles too.
# # This is why accurate physical size measurement matters.

In [ ]:
# ============================================================
# EXERCISE 2: Compute alignment offsets
# ============================================================
# For the multi-marker scene, compute:
#   - Which marker is closest (smallest Z)?
#   - What is the lateral offset (X, Y) for each marker?
# Print a report like:
#   ID=0: Z=45.2cm, X_offset=+2.1cm, Y_offset=-1.3cm
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# if i_multi is not None:
#     closest_id = None
#     closest_z = float('inf')
#     for mid, tvec in zip(i_multi.flatten(), tv_multi):
#         tx, ty, tz = tvec.flatten()
#         print(f'ID={mid}: Z={tz*100:.1f}cm, X_offset={tx*100:+.1f}cm, Y_offset={ty*100:+.1f}cm')
#         if tz < closest_z:
#             closest_z = tz
#             closest_id = mid
#     print(f'\nClosest marker: ID={closest_id} at Z={closest_z*100:.1f}cm')

In [ ]:
# ============================================================
# EXERCISE 3: Validate rvec with projectPoints
# ============================================================
# Use cv2.projectPoints to reproject the 4 known 3D corners of the marker
# back to 2D using the estimated rvec and tvec.
# Compare reprojected corners vs detected corners.
# Compute mean reprojection error in pixels.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# if ids is not None:
#     L = MARKER_LENGTH / 2
#     obj_pts_3d = np.array([
#         [-L,  L, 0],
#         [ L,  L, 0],
#         [ L, -L, 0],
#         [-L, -L, 0]], dtype=np.float64)
#
#     proj_pts, _ = cv2.projectPoints(obj_pts_3d, rvecs[0], tvecs[0], K, dist)
#     proj_pts = proj_pts.reshape(4, 2)
#     det_pts  = corners[0].reshape(4, 2)
#
#     errors = np.linalg.norm(proj_pts - det_pts, axis=1)
#     print('Corner reprojection errors (pixels):')
#     for j, e in enumerate(['TL','TR','BR','BL']):
#         print(f'  {e}: {errors[j]:.3f} px')
#     print(f'Mean error: {errors.mean():.3f} px')